# Day 2: Streaming Ingestion (Kafka / MSK + Spark Streaming)

This notebook simulates streaming ingestion of synthetic FHIR records into the Bronze Iceberg table.

In [ ]:
import json
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit, expr

spark = SparkSession.builder \
    .appName("Day02_Streaming_Ingestion") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "/tmp/iceberg_warehouse") \
    .getOrCreate()

print("Spark session initialized.")

In [ ]:
# Create synthetic data using memory rate source
rate_stream = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 1) \
    .load()

fhir_data = rate_stream \
    .withColumn("raw_payload_json", expr("concat('{\"resourceType\": \"Patient\", \"id\": \"', cast(value as string), '\"}')")) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_system_id", lit("memory_stream")) \
    .withColumn("payload_id", expr("uuid()"))

query = fhir_data.select("ingestion_timestamp", "source_system_id", "payload_id", "raw_payload_json") \
    .writeStream \
    .format("memory") \
    .queryName("bronze_stream") \
    .outputMode("append") \
    .start()

print("Started streaming query.")

In [ ]:
# Wait for a few seconds to accumulate data
time.sleep(5)
spark.sql("SELECT * FROM bronze_stream").show(truncate=False)

In [ ]:
query.stop()
print("Stopped streaming query.")